## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# a running log of every row dropped, printed at the end
drop_log = []

def log_drop(stage, before, after):
    dropped = before - after
    drop_log.append({
        "stage": stage,
        "dropped": dropped,
        "remaining": after,
        "pct_of_original": round(dropped / before * 100, 2)
    })
    print(f"{stage:45s} dropped {dropped:>7,}  remaining {after:>8,}")

In [2]:
ppr = pd.read_csv(RAW / "ppr_raw.csv", encoding="latin-1")
n_original = len(ppr)

print(f"Loaded {n_original:,} rows")
print(f"Columns: {list(ppr.columns)}")

Loaded 799,067 rows
Columns: ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (\x80)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']


C:\Users\heffo\AppData\Local\Temp\ipykernel_12464\3903689039.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ppr = pd.read_csv(RAW / "ppr_raw.csv", encoding="latin-1")


## 2. Rename and Parse

In [4]:
# Renaming columns

ppr = ppr.rename(columns={
    "Date of Sale (dd/mm/yyyy)":  "date",
    "Address":                    "address",
    "County":                     "county",
    "Eircode":                    "eircode",
    "Price (\x80)":               "price_raw",
    "Not Full Market Price":      "not_full_market",
    "VAT Exclusive":              "vat_exclusive",
    "Description of Property":    "description",
    "Property Size Description":  "size_description",
})

print(list(ppr.columns))

['date', 'address', 'county', 'eircode', 'price_raw', 'not_full_market', 'vat_exclusive', 'description', 'size_description']


In [5]:
# Parse price and date 

ppr["price"] = (ppr["price_raw"]
                .str.replace("\x80", "", regex=False)
                .str.replace(",", "", regex=False)
                .str.strip()
                .astype(float))

ppr["date"] = pd.to_datetime(ppr["date"], format="%d/%m/%Y")
ppr["year"] = ppr["date"].dt.year
ppr["month"] = ppr["date"].dt.to_period("M")

print(f"Date range: {ppr['date'].min().date()} to {ppr['date'].max().date()}")
print(f"\nPrice summary:")
print(ppr["price"].describe().apply(lambda v: f"{v:,.0f}").to_string())

Date range: 2010-01-01 to 2026-07-31

Price summary:
count        799,067
mean         319,174
std        1,234,377
min            5,001
25%          145,000
50%          243,500
75%          365,000
max      387,665,198


In [10]:
print("Ten highest:")
print(ppr.nlargest(10, "price")[["date", "address", "county", "price"]].to_string())

print("\nTen lowest:")
print(ppr.nsmallest(10, "price")[["date", "address", "county", "price"]].to_string())

Ten highest:
             date                                                                      address   county         price
690730 2024-10-17                                  24 Tinakilly Grove, Tinakilly Park, Rathnew  Wicklow  3.876652e+08
586434 2023-02-10                                     O'Devaney Gardens, Arbour Hill, Dublin 7   Dublin  2.250000e+08
771088 2026-01-27                                                         Montpelier, Dublin 7   Dublin  2.250000e+08
706188 2024-12-23                                       Cooper Square, Seven Mills, Clonburris   Dublin  2.219427e+08
780814 2026-03-31                                            BLOCK A  B AND C, NEWMARKET YARDS   Dublin  1.893925e+08
431243 2020-07-17  Apartments 1 - 186 Cheevers Court, Apartments 1-182 Haliday House, Cualanor   Dublin  1.823789e+08
473413 2021-04-15                                          8th Lock, Ratoath Road, Pelletstown   Dublin  1.701428e+08
749598 2025-09-30                     Block

In [11]:
print("Price distribution tails:")
for q in [0.0001, 0.001, 0.005, 0.01, 0.05, 0.5, 0.95, 0.99, 0.995, 0.999, 0.9999]:
    print(f"  {q:>8.2%}  {ppr['price'].quantile(q):>15,.0f}")

Price distribution tails:
     0.01%            5,714
     0.10%            8,000
     0.50%           16,667
     1.00%           24,000
     5.00%           50,100
    50.00%          243,500
    95.00%          700,000
    99.00%        1,412,500
    99.50%        2,000,000
    99.90%        6,239,804
    99.99%       50,690,876


In [13]:
extremes = ppr[(ppr["price"] < 20_000) | (ppr["price"] > 2_000_000)]
print(f"Rows outside 20k - 2m: {len(extremes):,}  ({len(extremes)/len(ppr)*100:.2f}%)")
print(f"\nOf which flagged not-full-market:")
print(extremes["not_full_market"].value_counts().to_string())

print(f"\nSplit by tail:")
low = ppr[ppr["price"] < 20_000]
high = ppr[ppr["price"] > 2_000_000]
print(f"  below 20k:  {len(low):,}  ({(low['not_full_market']=='Yes').sum():,} flagged)")
print(f"  above 2m:   {len(high):,}  ({(high['not_full_market']=='Yes').sum():,} flagged)")

Rows outside 20k - 2m: 8,836  (1.11%)

Of which flagged not-full-market:
not_full_market
No     7603
Yes    1233

Split by tail:
  below 20k:  4,888  (1,029 flagged)
  above 2m:   3,948  (204 flagged)


## 3. Covert Irish language to English

In [8]:
print("Before mapping:")
print(ppr["description"].value_counts(dropna=False).to_string())

Before mapping:
description
Second-Hand Dwelling house /Apartment    656283
New Dwelling house /Apartment            142735
Teach/Árasán Cónaithe Atháimhe               45
Teach/Árasán Cónaithe Nua                     3
Teach/?ras?n C?naithe Nua                     1


In [7]:
# anything containing "Nua" (new) or the mojibake equivalent is a new dwelling,
# anything containing "Atháimhe" / "Ath" is second-hand

def map_description(val):
    v = str(val)
    if "New Dwelling" in v or "Nua" in v:
        return "New"
    if "Second-Hand" in v or "Ath" in v or "th\u00e1imhe" in v:
        return "Second-Hand"
    return np.nan

ppr["property_type"] = ppr["description"].apply(map_description)

print("After mapping:")
print(ppr["property_type"].value_counts(dropna=False).to_string())

After mapping:
property_type
Second-Hand    656328
New            142739


## 4. VAT Adjustment

In [9]:
# new residential property prices in the PPR are filed excluding VAT
# Irish VAT on new residential property is 13.5%

VAT_RATE = 0.135

ppr["price_incl_vat"] = np.where(
    ppr["vat_exclusive"] == "Yes",
    ppr["price"] * (1 + VAT_RATE),
    ppr["price"]
)

n_adjusted = (ppr["vat_exclusive"] == "Yes").sum()
print(f"VAT adjustment applied to {n_adjusted:,} sales")

comparison = ppr.groupby("vat_exclusive").agg(
    n=("price", "size"),
    median_before=("price", "median"),
    median_after=("price_incl_vat", "median"),
)
print(f"\n{comparison.to_string()}")

VAT adjustment applied to 140,374 sales

                    n  median_before  median_after
vat_exclusive                                     
No             658693       226000.0    226000.000
Yes            140374       308369.0    349998.815


In [14]:
print(pd.crosstab(ppr["property_type"], ppr["vat_exclusive"]))

vat_exclusive      No     Yes
property_type                
New              2365  140374
Second-Hand    656328       0


In [15]:
new_only = ppr[ppr["property_type"] == "New"]
print(new_only.groupby("vat_exclusive")["price"].describe()[["count", "25%", "50%", "75%"]].round(0).to_string())

                  count       25%       50%       75%
vat_exclusive                                        
No               2365.0   80000.0  158546.0  350000.0
Yes            140374.0  220264.0  308369.0  396476.0


2,365 sales are described as new dwellings but not flagged VAT-exclusive (1.7% of new builds). Their price distribution sits well below VAT-exclusive new builds rather than above, so these are unlikely to be gross-priced new sales. They are more plausibly non-standard transactions such as local authority or affordable housing transfers. No VAT adjustment is applied, consistent with keying the adjustment off the VAT flag rather than the property description.

## 5. Filtering

In [19]:
n = len(ppr)

# 1. non-arms-length transactions
ppr = ppr[ppr["not_full_market"] == "No"].copy()
log_drop("Not full market price", n, len(ppr)); n = len(ppr)

# 2. restrict to individual dwellings
#    upper bound removes bulk and portfolio transactions
#    lower bound removes nominal transfers
PRICE_MIN, PRICE_MAX = 20_000, 2_000_000
ppr = ppr[ppr["price_incl_vat"].between(PRICE_MIN, PRICE_MAX)].copy()
log_drop(f"Price outside {PRICE_MIN:,} - {PRICE_MAX:,}", n, len(ppr)); n = len(ppr)

# 3. placeholder Eircodes - blank the code, keep the sale
PLACEHOLDERS = ["A123456", "A00AA00"]
mask = ppr["eircode"].isin(PLACEHOLDERS)
print(f"\nPlaceholder Eircodes blanked: {mask.sum():,}")
ppr.loc[mask, "eircode"] = np.nan

Not full market price                         dropped  40,721  remaining  758,346
Price outside 20,000 - 2,000,000              dropped   7,686  remaining  750,660

Placeholder Eircodes blanked: 154


In [20]:
survivors = ppr[(ppr["property_type"] == "New") & (ppr["vat_exclusive"] == "No")]
print(f"Remaining after filtering: {len(survivors):,}")

Remaining after filtering: 1,901


In [21]:
print(f"Rows now: {len(ppr):,}")
print(f"Drop log entries: {len(drop_log)}")

Rows now: 750,660
Drop log entries: 2


In [22]:
print(pd.DataFrame(drop_log).to_string(index=False))

                           stage  dropped  remaining  pct_of_original
           Not full market price    40721     758346             5.10
Price outside 20,000 - 2,000,000     7686     750660             1.01
